[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github.com/MLinApp-polito/mla-prj-23-project-am04_group-am01/blob/main/defect_detection.ipynb)

TODO: fix the button

# PBF Defect Detection

This notebook covers data loading, training, validation, and inference for detecting defects in Powder Bed Fusion images using a fine-tuned CNN.

## Clone GithHub repo

In [56]:
!rm -rf mla_project/

In [ ]:
import os

if not os.path.exists("/content/mla-prj-23-project-am04_group-am01") and not os.path.exists("/content/mla_project"):
  # DON'T SHARE THE PERSONAL ACCESS TOKEN

  # change the name of the branch here as needed
  !git clone -b singan https://***REMOVED-GITHUB-TOKEN***@github.com/MLinApp-polito/mla-prj-23-project-am04_group-am01.git

  # Rename folder for simplicity
  !mv /content/mla-prj-23-project-am04_group-am01 /content/mla_project

!cd /content/mla_project && git pull

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 5 (delta 3), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 411 bytes | 205.00 KiB/s, done.
From https://github.com/MLinApp-polito/mla-prj-23-project-am04_group-am01
   1dbb6d3..5cd3c15  giacomo    -> origin/giacomo
Updating 1dbb6d3..5cd3c15
Fast-forward
 external/PyTorch-SinGAN/random_samples.py | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)


## Install Dependencies

In [3]:
!pip install torch torchvision matplotlib tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 83.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

## Imports

In [4]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# Original dataset

## Dataset mean and std

In [ ]:
!python /content/mla_project/src/data_loader.py --data-dir /content/mla_project/images/original --compute-stats

Dataset mean (grayscale): 0.5839
Dataset std (grayscale): 0.2074


## Training - no augmentation

**IMPORTANT:**

- To perform K-Fold cross-validation, set --is_kfold to "True" and specify the number of folds with --k-folds. Example: --is_kfold "True", --k-folds 5

- To perform a single train/val split, set --is_kfold to "False" and specify the validation split ratio with --val-split. Example: --is_kfold "False", --val-split 0.2

- In both cases, to perform also testing, set --test to "True" and specify the test split ratio with --test-split. Example: --test "True", --test-split 0.2

### Define paths and parameters

In [ ]:
data_dir = '/content/mla_project/images/original'

# Training params
batch_size = 2
epochs = 10
learning_rate = 1e-5
backbone = 'resnet50'
device = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


### Launch training

In [ ]:
# k-fold cross validation (with test)

!python /content/mla_project/src/train.py \
    --data-dir "{data_dir}" \
    --batch-size {batch_size} \
    --epochs {10} \
    --lr {learning_rate} \
    --backbone {backbone} \
    --num-workers 2 \
    --is_kfold "True" \
    --k-folds 5 \
    --test "True" \
    --test-split 0.2

### Plot Training & Validation Curves

In [ ]:
# plot for cross-validation
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Read logs
logs = pd.read_csv('/content/kfold_logs.csv')

# Group for epoch and get mean and std
grouped = logs.groupby('epoch').agg({
    'train_loss': ['mean', 'std'],
    'val_loss': ['mean', 'std'],
    'train_acc': ['mean', 'std'],
    'val_acc': ['mean', 'std']
}).reset_index()

# Rename columns
grouped.columns = ['epoch',
                   'train_loss_mean', 'train_loss_std',
                   'val_loss_mean', 'val_loss_std',
                   'train_acc_mean', 'train_acc_std',
                   'val_acc_mean', 'val_acc_std']

# Set style
sns.set(style="white", context="notebook")

# LOSS
plt.figure(figsize=(8, 5))
plt.plot(grouped['epoch'], grouped['train_loss_mean'], label='Train Loss', color='blue')
plt.fill_between(grouped['epoch'],
                 grouped['train_loss_mean'] - grouped['train_loss_std'],
                 grouped['train_loss_mean'] + grouped['train_loss_std'],
                 color='blue', alpha=0.2)

plt.plot(grouped['epoch'], grouped['val_loss_mean'], label='Val Loss', color='orange')
plt.fill_between(grouped['epoch'],
                 grouped['val_loss_mean'] - grouped['val_loss_std'],
                 grouped['val_loss_mean'] + grouped['val_loss_std'],
                 color='orange', alpha=0.2)

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss (mean ± std)')
plt.legend()
sns.despine()
plt.tight_layout()
plt.show()

# ACCURACY
plt.figure(figsize=(8, 5))
plt.plot(grouped['epoch'], grouped['train_acc_mean'], label='Train Accuracy', color='green')
plt.fill_between(grouped['epoch'],
                 grouped['train_acc_mean'] - grouped['train_acc_std'],
                 grouped['train_acc_mean'] + grouped['train_acc_std'],
                 color='green', alpha=0.2)

plt.plot(grouped['epoch'], grouped['val_acc_mean'], label='Val Accuracy', color='red')
plt.fill_between(grouped['epoch'],
                 grouped['val_acc_mean'] - grouped['val_acc_std'],
                 grouped['val_acc_mean'] + grouped['val_acc_std'],
                 color='red', alpha=0.2)

plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy (mean ± std)')
plt.legend()
sns.despine()
plt.tight_layout()
plt.show()


## Training - basic augmentations (simple transformations)

In [ ]:
data_dir = '/content/mla_project/images/original'

# Training params
batch_size = 2
epochs = 10
learning_rate = 1e-5
backbone = 'resnet50'
device = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# k-fold cross validation (with test) with augmented data

!python /content/mla_project/src/train.py \
    --data-dir "{data_dir}" \
    --batch-size {batch_size} \
    --epochs {10} \
    --lr {learning_rate} \
    --backbone {backbone} \
    --num-workers 2 \
    --is_kfold "True" \
    --k-folds 5 \
    --test "True" \
    --test-split 0.2 \
    --aug "True"

In [ ]:
# plot for cross-validation
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Read logs
logs = pd.read_csv('/content/kfold_logs.csv')

# Group for epoch and get mean and std
grouped = logs.groupby('epoch').agg({
    'train_loss': ['mean', 'std'],
    'val_loss': ['mean', 'std'],
    'train_acc': ['mean', 'std'],
    'val_acc': ['mean', 'std']
}).reset_index()

# Rename columns
grouped.columns = ['epoch',
                   'train_loss_mean', 'train_loss_std',
                   'val_loss_mean', 'val_loss_std',
                   'train_acc_mean', 'train_acc_std',
                   'val_acc_mean', 'val_acc_std']

# Set style
sns.set(style="white", context="notebook")

# LOSS
plt.figure(figsize=(8, 5))
plt.plot(grouped['epoch'], grouped['train_loss_mean'], label='Train Loss', color='blue')
plt.fill_between(grouped['epoch'],
                 grouped['train_loss_mean'] - grouped['train_loss_std'],
                 grouped['train_loss_mean'] + grouped['train_loss_std'],
                 color='blue', alpha=0.2)

plt.plot(grouped['epoch'], grouped['val_loss_mean'], label='Val Loss', color='orange')
plt.fill_between(grouped['epoch'],
                 grouped['val_loss_mean'] - grouped['val_loss_std'],
                 grouped['val_loss_mean'] + grouped['val_loss_std'],
                 color='orange', alpha=0.2)

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss (mean ± std)')
plt.legend()
sns.despine()
plt.tight_layout()
plt.show()

# ACCURACY
plt.figure(figsize=(8, 5))
plt.plot(grouped['epoch'], grouped['train_acc_mean'], label='Train Accuracy', color='green')
plt.fill_between(grouped['epoch'],
                 grouped['train_acc_mean'] - grouped['train_acc_std'],
                 grouped['train_acc_mean'] + grouped['train_acc_std'],
                 color='green', alpha=0.2)

plt.plot(grouped['epoch'], grouped['val_acc_mean'], label='Val Accuracy', color='red')
plt.fill_between(grouped['epoch'],
                 grouped['val_acc_mean'] - grouped['val_acc_std'],
                 grouped['val_acc_mean'] + grouped['val_acc_std'],
                 color='red', alpha=0.2)

plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy (mean ± std)')
plt.legend()
sns.despine()
plt.tight_layout()
plt.show()

# Generative Adversarial Networks

## Training - GANs

### SinGAN

In [61]:
# Reset folders
!rm -rf /content/SinGANRandomSamples/
!rm -rf /content/SinGANTrainedModels/


In [62]:
input_dir = "/content/mla_project/images/original"

#### Train SinGAN

In this phase, a **SinGAN** model is trained using a single image selected by the user from the original dataset.

The major parameters that we can modify to obtain a better model are the following (with the corrisponding default values):
- scale_factor 0.75 
- niter 2000
- lr_g 0.0005
- lr_d 0.0005 
- Gsteps 5
- Dsteps 5

In [ ]:
# Set parameters for training
class_ = "Defects"
image_name = "Image0.jpg"
mode = 'train'
scale_factor = 0.80
niter = 150
lr_g = 0.001
lr_d = 0.0005
G_steps = 7
D_steps = 5

# Train the SinGAN model on the specified image
!python /content/mla_project/external/SinGAN/main_train.py  --input_dir "{input_dir}/{class_}/" \
                                                            --class_ "{class_}" \
                                                            --input_name "{image_name}" \
                                                            --mode '{mode}' \
                                                            --scale_factor {scale_factor} \
                                                            --niter {niter} \
                                                            --lr_g {lr_g} \
                                                            --lr_d {lr_d} \
                                                            --Gsteps {G_steps} \
                                                            --Dsteps {D_steps} 

Image size: 410 x 512
R1 penalty is not used
Number of scales: 3
stop_scale: 3
Image size at scale 0: 11x13
scale 0:[0/1] - errD: -1.5346, errG: 1.0623
Image size at scale 1: 35x44
scale 1:[0/1] - errD: -0.1986, errG: 0.0720
Image size at scale 2: 119x149
scale 2:[0/1] - errD: -0.0186, errG: -0.1290
Image size at scale 3: 410x512
scale 3:[0/1] - errD: -0.0213, errG: -0.1183


#### Create trained models

In this phase, the trained models are created and compressed into a ZIP archive for easier storage and sharing.


In [ ]:
# Compress the entire TrainedModels directory into a ZIP file named trained_models.zip
!zip trained_models.zip /content/SinGANTrainedModels/

# Compress the modles of a specific image and class
class_zip = "Defects"
image_name_zip = "Image0"
#!zip trained_models_{class_zip}_{image_name_zip}.zip /content/SinGANTrainedModels/{class_zip}/{image_name_zip}

#### Generate new images using SinGAN

In this phase, random samples are generated using the SinGAN model for the specified image and class.

By default, SinGAN will use the most recently trained model for the selected image.  
However, it is also possible to specify a particular model by providing the name of the folder containing it (e.g., `"scale_factor=0.800000,alpha=10"`).  
This folder must be located inside the "`TrainedModels`" directory for the chosen image.

To use this model, uncomment and set the `--model_name` parameter accordingly.

NB:

Do not modify the `--gen_start_scale 0` parameter, otherwise it will not be able to generate the images with all the learned details.


In [ ]:
# Remove any previously generated samples for this specific image and class
!rm -rf "/content/SinGANRandomSamples/{class_samples}/{image_samples_without_ext}"

In [ ]:
class_samples = "Defects"          # Class label or category of the image
image_name_samples = "Image0.jpg"   # Filename of the input image to be used for generate samples

# Optional
name_model_trained = "scale_factor=0.800000,alpha=10" # Name of the folder that contains the trained model

# Extract the base name of the image (without file extension)
image_samples_without_ext = image_name_samples.split('.')[0]

# Generate random samples using the SinGAN model for the specified image and class
!python /content/mla_project/external/SinGAN/random_samples.py --input_dir "{input_dir}/{class_samples}/" \
                                                      --class_ "{class_samples}" \
                                                      --input_name "{image_name_samples}" \
                                                      --mode random_samples \
                                                      --scale_factor {scale_factor} \
                                                      --gen_start_scale 0 \
                                                      --dir_model "{name_model_trained}" \
                                                      --not_cuda # if you not have the GPU (good luck)

Generating random sample 0/50 for image Image0
Generating random sample 1/50 for image Image0
Generating random sample 2/50 for image Image0
Generating random sample 3/50 for image Image0
Generating random sample 4/50 for image Image0
Generating random sample 5/50 for image Image0
Generating random sample 6/50 for image Image0
Generating random sample 7/50 for image Image0
Generating random sample 8/50 for image Image0
Generating random sample 9/50 for image Image0
Generating random sample 10/50 for image Image0
Generating random sample 11/50 for image Image0
Generating random sample 12/50 for image Image0
Generating random sample 13/50 for image Image0
Generating random sample 14/50 for image Image0
Generating random sample 15/50 for image Image0
Generating random sample 16/50 for image Image0
Generating random sample 17/50 for image Image0
Generating random sample 18/50 for image Image0
Generating random sample 19/50 for image Image0
Generating random sample 20/50 for image Image0
Ge

#### Compress Random Samples

In this phase, all generated random samples are compressed into a ZIP file, both for all images and for a specific image.

In [79]:
# Zip all generated random samples of all images into a file named random_samples.zip
!zip -r random_samples.zip /content/SinGANRandomSamples/
print("\n")

# Zip all generated random samples of a specific image
class_zip = "Defects"
image_zip = "Image0"
#!zip -r random_samples_{class_zip}_{image_zip}.zip /content/SinGANRandomSamples/{class_zip}/{image_zip}

updating: content/RandomSamples/ (stored 0%)
updating: content/RandomSamples/Defects/ (stored 0%)
updating: content/RandomSamples/Defects/Image0/ (stored 0%)
updating: content/RandomSamples/Defects/Image0/scale_factor=0.100000,alpha=10/ (stored 0%)
updating: content/RandomSamples/Defects/Image0/scale_factor=0.100000,alpha=10/Image0_29.png (deflated 1%)
updating: content/RandomSamples/Defects/Image0/scale_factor=0.100000,alpha=10/Image0_3.png (deflated 1%)
updating: content/RandomSamples/Defects/Image0/scale_factor=0.100000,alpha=10/Image0_31.png (deflated 1%)
updating: content/RandomSamples/Defects/Image0/scale_factor=0.100000,alpha=10/Image0_22.png (deflated 1%)
updating: content/RandomSamples/Defects/Image0/scale_factor=0.100000,alpha=10/Image0_35.png (deflated 1%)
updating: content/RandomSamples/Defects/Image0/scale_factor=0.100000,alpha=10/Image0_34.png (deflated 1%)
updating: content/RandomSamples/Defects/Image0/scale_factor=0.100000,alpha=10/Image0_6.png (deflated 1%)
updating: c